In [ ]:
# !pip install google-api-python-client pandas openpyxl -q

In [ ]:
import re
import time
import pandas as pd
from googleapiclient.discovery import build

In [ ]:
import os

# =========================
# 1. YouTube API 설정
# =========================
API_KEY = os.environ["YOUTUBE_API_KEY"]
youtube = build("youtube", "v3", developerKey=API_KEY)

In [ ]:
# =========================
# 2. 첼시 관련 유튜브 영상 URL 목록
# =========================
video_urls = [
    "https://www.youtube.com/watch?v=15pAXEqG878", #38라운드 (선덜랜드)
    "https://www.youtube.com/watch?v=wgz8Lv5ritw", #37라운드 (토트넘)
    "https://www.youtube.com/watch?v=x2cU8_DFCXk", #36라운드 (리버풀)
    "https://www.youtube.com/watch?v=GhmIsk5U7c4", #35라운드 (노팅엄)
    "https://www.youtube.com/watch?v=oFtYYLHcnI8", #34라운드 (브라이튼)
    "https://www.youtube.com/watch?v=uXX3KyajIIQ", #33라운드 (맨유)
    "https://www.youtube.com/watch?v=kP_8XNSepX8", #32라운드 (맨시티)
    "https://www.youtube.com/watch?v=6yWJ-AF5Xlk", #31라운드 (에버턴)
    "https://www.youtube.com/watch?v=QpS_KVXgths", #30라운드 (뉴캐슬)
    "https://www.youtube.com/watch?v=GmgFr1boMek", #29라운드 (아스톤빌라)
]

MAX_COMMENTS_PER_VIDEO = 1000

def extract_video_id(url):
    match = re.search(r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})", url)
    return match.group(1) if match else None

def clean_comment(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:

# =========================
# 3. 댓글 수집
# =========================
all_comments = []
seen_comment_ids = set()

print("수집할 영상 개수:", len(video_urls))
print("=" * 60)

for idx, url in enumerate(video_urls, start=1):
    video_id = extract_video_id(url)
    if not video_id:
        print(f"[건너뜀] video_id를 추출할 수 없음: {url}")
        continue

    print(f"\n[{idx}/{len(video_urls)}] 댓글 수집 시작: {video_id}")
    page_count = 0
    video_comment_count = 0
    next_page_token = None

    try:
        while True:
            request = youtube.commentThreads().list(part="snippet", videoId=video_id, maxResults=100, textFormat="plainText", order="relevance", pageToken=next_page_token)
            response = request.execute()
            page_count += 1
            items = response.get("items", [])
            new_count_this_page = 0

            for item in items:
                comment = item["snippet"]["topLevelComment"]
                comment_id = comment["id"]
                if comment_id in seen_comment_ids:
                    continue
                seen_comment_ids.add(comment_id)
                comment_data = comment["snippet"]
                original_comment = comment_data.get("textDisplay", "")
                clean_text = clean_comment(original_comment)
                all_comments.append({"comment_id": comment_id, "video_id": video_id, "video_url": url, "author": comment_data.get("authorDisplayName"), "comment_original": original_comment, "comment_clean": clean_text, "like_count": comment_data.get("likeCount"), "published_at": comment_data.get("publishedAt")})
                video_comment_count += 1
                new_count_this_page += 1

            print(f"[진행중] 영상 {video_id} | {page_count}페이지 완료 | 이번 페이지 신규 {new_count_this_page}개 | 현재 {video_comment_count}개 수집")

            if new_count_this_page == 0:
                print(f"[중단] 영상 {video_id} | 신규 댓글 0개라 반복 수집 방지 중단")
                break
            if video_comment_count >= MAX_COMMENTS_PER_VIDEO:
                print(f"[중단] 영상 {video_id} | {MAX_COMMENTS_PER_VIDEO}개 도달")
                break
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
            time.sleep(0.1)
        print(f"[완료] 영상 {video_id} | 총 {video_comment_count}개 수집")
    except Exception as e:
        print(f"[에러 발생] 영상 {video_id}")
        print(e)
        continue

In [ ]:
# =========================
# 4. 데이터프레임 생성 및 저장
# =========================
df = pd.DataFrame(all_comments)
print("\n" + "=" * 60)
print("전체 원본 수집 댓글 수:", len(df))

if len(df) > 0:
    df = df[df["comment_clean"].str.len() > 0].copy()
    df = df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    print("중복 제거 후 댓글 수:", len(df))
    df.to_csv("chelsea_youtube_comments_clean.csv", index=False, encoding="utf-8-sig")
    df.to_excel("chelsea_youtube_comments_clean.xlsx", index=False)
    print("저장 완료")
    print("CSV 파일: chelsea_youtube_comments_clean.csv")
    print("Excel 파일: chelsea_youtube_comments_clean.xlsx")
else:
    print("수집된 댓글이 없습니다.")

df.head()


# GPT API 1차 감정분석


In [ ]:
# !pip install openai tqdm

In [ ]:
import pandas as pd
import json
import time
from tqdm import tqdm
from openai import OpenAI

client = OpenAI()

# =========================
# 1. 댓글 파일 불러오기
# =========================
df = pd.read_csv("chelsea_youtube_comments_clean.csv")
print("전체 댓글 수:", len(df))
df.head()


In [ ]:
# =========================
# 2. 감정분석 함수
# =========================
def analyze_sentiment(comment):
    prompt = f"""
너는 한국어 축구 팬덤 댓글을 분석하는 감정분석 분류기다.

분석 대상은 EPL 첼시 FC 관련 유튜브 댓글이다.

분석의 목표는 댓글이 첼시, 첼시 선수, 첼시 감독, 경기력, 우승 경쟁, 이적시장, 전술, 구단 운영, 팬덤 분위기에 대해 긍정적인지, 부정적인지, 중립적인지 분류하는 것이다.

[첼시 선수단 정보]
현재 감독: 맥팔레인(임시 감독)
이전 감독: 로세니어, 마레스카

선수:
로베르트 산체스, 쿠쿠레야(쿠쿠렐라), 토신(토신 아다라비요), 바디아실, 콜윌, 페드루 네트, 엔소 페르난데스,
리암 델랍, 콜 파머, 제이미 기튼스, 요르겐센, 다리우 이수구, 안드레이 산투스, 마마두 사르, 주앙 페드루,
조렐 하토, 트레보 찰로바, 리스 제임스(리제), 카이세도, 귀스토(구스토), 포파나, 아체암퐁, 마르크 기우,
우드먼, 프림퐁, 흐라벤베르흐(그라벤베르흐), 트레이 뇨니, 리오 은구모하, 이스테방, 라비아, 가르나초

[분류 기준]

1. positive

다음 중 하나에 해당하면 positive로 분류한다.

* 첼시 팀, 감독, 선수, 전술, 경기 결과를 긍정적으로 평가함
* 첼시 선수의 실력, 성장, 득점, 도움, 수비, 압박, 리더십을 칭찬함
* 첼시의 승리, 연승, 순위 상승, 우승 가능성, 미래 가능성을 긍정적으로 언급함
* 첼시 팬덤을 긍정적으로 평가함
* 첼시 유망주, 영입 선수, 감독에 대해 기대감을 표현함
* 비속어나 농담이 포함되어도 실제 의미가 첼시 칭찬이면 positive

예시

* 첼시 경기력 많이 좋아졌다
* 팔머는 진짜 월클이다
* 카이세도 오늘 미쳤다
* 로세니어 전술 괜찮네
* 첼시 미래가 밝다
* 이번 시즌 기대된다
* 블루스 돌아왔다
* 첼시 다시 강해진다

2. negative

다음 중 하나에 해당하면 negative로 분류한다.

* 첼시 팀, 감독, 선수, 전술, 경기 결과를 부정적으로 평가함
* 첼시 선수의 실수, 부진, 결정력 부족, 수비 실책, 태도 문제를 비판함
* 첼시의 패배, 순위 하락, 우승 실패, 경기력 저하를 비판함
* 조롱, 비난, 실망, 분노가 첼시를 향함
* 첼시 선수의 실수나 부진을 웃음거리로 만드는 경우
* 첼시의 이적 정책, 구단 운영, 구단주를 비판하는 경우

예시

* 첼시 오늘 왜 이렇게 못하냐
* 공격진 결정력 진짜 최악이다
* 델랍 골결이 진짜 별로다
* 수비가 너무 불안하다
* 전술이 답이 없다
* 이러면 챔스도 못가지
* 또 실수하네
* 돈은 돈대로 쓰고 결과는 없네
* 경기력 처참하다

3. neutral

다음 중 하나에 해당하면 neutral로 분류한다.

* 경기 일정 질문
* 선수 정보 질문
* 단순 사실 전달
* 첼시와 직접 관련 없는 내용
* 감정 방향이 전혀 드러나지 않음
* 맥락 파악이 불가능함

예시

* 오늘 경기 몇 시임?
* 팔머 나이 몇 살임?
* 첼시 다음 경기 언제임?
* EPL 일정 빡세네
* 프리미어리그 재밌다

[첼시 팬덤 멸칭 및 조롱 표현 규칙]

다음 표현은 첼시를 조롱하거나 비하하는 표현으로 간주한다.

아래 단어 또는 유사 표현이 포함되면 sentiment를 negative로 분류한다.

* 첼강딱
* 돈시
* 첼레반
* 첼시놈
* 보엘리 FC
* 1조 FC
* 돈지랄 FC
* 강등시
* 돈은 돈대로 쓰고
* 돈 처바르고
* 천억 쓰고
* 1조 쓰고

예시

* 첼강딱 수준
* 돈시 또 시작이네
* 1조 쓰고 이게 맞냐
* 보엘리 FC 레전드
* 돈은 돈대로 쓰고 경기력은 최악

위 표현들은 첼시에 대한 조롱 및 비하 표현으로 간주하며 negative로 분류한다.

[중요한 축구 팬덤 표현 해석]

positive 표현 예시

* GOAT
* 월클
* 클래스
* 레전드
* 미쳤다
* 지린다
* 개잘한다
* 폼 미쳤다
* 캐리
* 미친 경기력
* 황제
* 킹
* goat다

negative 표현 예시

* 개못한다
* 답답하다
* 망했다
* 처참하다
* 최악
* 왜 저러냐
* 노답
* 심각하다
* 개판이다
* 실망이다
* 나가라
* 경질
* 방출
* 은퇴해라
* 폼 다 죽었다

판단 원칙

* 비속어가 포함되어도 실제 의미가 칭찬이면 positive
* 비속어가 포함되어도 실제 의미가 비판이면 negative
* 상대팀 칭찬만 하는 경우는 neutral
* 상대팀 칭찬을 통해 첼시를 깎아내리면 negative
* 첼시를 조롱하거나 비꼬면 negative
* 첼시 팬의 실망, 분노, 불안은 negative

[조롱성 웃음 표현 규칙]

다음 표현이 포함되어 있더라도

* ㅋㅋ
* ㅋㅋㅋ
* 개웃기네
* 웃음벨
* 과학
* 레전드
* 개꿀잼
* 역대급

첼시 또는 첼시 선수의

* 실수
* 패배
* 부진
* 결정력 부족
* 수비 실책
* 경기력 저하

를 비웃는 맥락이면 반드시 negative로 분류한다.

예시

* 또 졌네 ㅋㅋㅋ
* 델랍 레전드네 ㅋㅋ
* 돈 쓰고 이 꼴이네
* 보엘리 축구 개웃기네

[neutral 최소화 규칙]

다음 경우는 neutral이 아니다.

* 선수 이름 + 칭찬 표현 → positive
* 선수 이름 + 비판 표현 → negative
* 감독 이름 + 칭찬 표현 → positive
* 감독 이름 + 비판 표현 → negative
* 경기력 언급 + 긍정 평가 → positive
* 경기력 언급 + 부정 평가 → negative
* 우승 가능성 언급 → positive
* 패배, 부진, 실망 언급 → negative

neutral은 정말 감정 방향을 판단할 수 없는 경우에만 사용한다.

[target 분류 기준]

* chelsea_team : 첼시 팀 전체, 구단, 팬덤, 우승, 순위, 경기 결과
* chelsea_player : 첼시 선수 개인
* chelsea_manager : 첼시 감독, 코칭스태프, 전술 운영
* opponent : 상대 팀 또는 상대 선수
* referee_or_league : 심판, 판정, EPL, UEFA 등 대회 운영
* unclear : 대상이 불명확한 경우

[판단 우선순위]

1. 댓글의 감정 대상이 첼시 팀, 선수, 감독인지 확인한다.
2. 첼시 관련 대상이면 positive 또는 negative를 우선 검토한다.
3. 선수 이름과 평가 표현이 있으면 neutral을 사용하지 않는다.
4. 축구 커뮤니티 특유의 칭찬 또는 비판 표현이 있으면 neutral보다 positive 또는 negative를 우선 선택한다.
5. 정말 감정을 판단할 수 없는 경우에만 neutral을 사용한다.

[출력 규칙]

반드시 아래 JSON 형식으로만 답하라.

설명 문장, 코드블록, 마크다운은 쓰지 마라.

{{
  "sentiment": "positive 또는 negative 또는 neutral",
  "confidence": 0.0,
  "target": "chelsea_team 또는 chelsea_player 또는 chelsea_manager 또는 opponent 또는 referee_or_league 또는 unclear",
  "reason": "짧은 한국어 이유"
}}

[분석할 댓글]

{comment}

"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "너는 한국어 첼시 유튜브 댓글 감정분석 전문가다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        result_text = response.choices[0].message.content.strip()
        result = json.loads(result_text)
        return (result.get("sentiment", "error"), result.get("confidence", 0.0), result.get("target", "unclear"), result.get("reason", ""))
    except Exception as e:
        return "error", 0.0, "error", str(e)


In [ ]:
sentiments = []
confidences = []
targets = []
reasons = []

for comment in tqdm(df["comment_clean"]):
    sentiment, confidence, target, reason = analyze_sentiment(comment)
    sentiments.append(sentiment)
    confidences.append(confidence)
    targets.append(target)
    reasons.append(reason)
    time.sleep(0.2)

df["gpt_sentiment"] = sentiments
df["gpt_confidence"] = confidences
df["gpt_target"] = targets
df["gpt_reason"] = reasons

df[["comment_clean", "gpt_sentiment", "gpt_confidence", "gpt_target", "gpt_reason"]]


In [ ]:
df.to_csv("chelsea_youtube_comments_clean_감정분석.csv", index=False, encoding="utf-8-sig")
df.to_excel("chelsea_youtube_comments_clean_감정분석.xlsx", index=False)